# Attention From Scratch

After reading Vaswani et al. properly, I want to implement the whole thing myself — scaled dot-product attention, then multi-head, then a full encoder block.

No HuggingFace. No shortcuts. Just PyTorch tensors and the equations from the paper.

I'll verify my output shapes match what you'd expect, and at the end compare to `nn.MultiheadAttention`.

## The Math (from the paper)

### Scaled Dot-Product Attention
```
Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) · V
```

- Q: queries  — shape (batch, seq_len, d_k)
- K: keys     — shape (batch, seq_len, d_k)
- V: values   — shape (batch, seq_len, d_v)
- Output      — shape (batch, seq_len, d_v)

### Multi-Head Attention
```
MultiHead(Q, K, V) = Concat(head_1, ..., head_h) · W_O
where head_i = Attention(Q·W_Qi, K·W_Ki, V·W_Vi)
```

Each head projects into a lower-dimensional space (d_model / h), runs attention, and they're concatenated back.

### Why sqrt(d_k)?
If Q and K entries have variance 1, then QK^T entries have variance d_k (sum of d_k products). Large values push softmax into near-zero gradient regions. Dividing by sqrt(d_k) brings variance back to 1.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

## Scaled Dot-Product Attention

In [2]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q, K, V: (batch, heads, seq_len, d_k)
    mask:    (batch, 1, 1, seq_len) — optional, for causal/padding masking
    """
    d_k = Q.shape[-1]

    # (batch, heads, seq_len, seq_len)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    attn_weights = F.softmax(scores, dim=-1)

    # (batch, heads, seq_len, d_v)
    output = torch.matmul(attn_weights, V)

    return output, attn_weights


# quick sanity check
batch, heads, seq_len, d_k = 2, 4, 10, 16
Q = torch.randn(batch, heads, seq_len, d_k)
K = torch.randn(batch, heads, seq_len, d_k)
V = torch.randn(batch, heads, seq_len, d_k)

out, weights = scaled_dot_product_attention(Q, K, V)
print(f"output shape:  {out.shape}")      # (2, 4, 10, 16)
print(f"weights shape: {weights.shape}")  # (2, 4, 10, 10)
print(f"weights sum to 1: {weights.sum(dim=-1).allclose(torch.ones(batch, heads, seq_len))}")

output shape:  torch.Size([2, 4, 10, 16])
weights shape: torch.Size([2, 4, 10, 10])
weights sum to 1: True


## Multi-Head Attention

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model    = d_model
        self.num_heads  = num_heads
        self.d_k        = d_model // num_heads

        # projection matrices — one for Q, K, V and one output
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def split_heads(self, x):
        """(batch, seq_len, d_model) → (batch, heads, seq_len, d_k)"""
        batch, seq_len, _ = x.shape
        x = x.view(batch, seq_len, self.num_heads, self.d_k)
        return x.transpose(1, 2)

    def combine_heads(self, x):
        """(batch, heads, seq_len, d_k) → (batch, seq_len, d_model)"""
        batch, _, seq_len, _ = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.view(batch, seq_len, self.d_model)

    def forward(self, Q, K, V, mask=None):
        # project
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))

        # attend
        attn_out, attn_weights = scaled_dot_product_attention(Q, K, V, mask)

        # combine and project
        out = self.W_o(self.combine_heads(attn_out))
        return out, attn_weights


# sanity check
d_model, num_heads, seq_len, batch = 512, 8, 20, 4
mha = MultiHeadAttention(d_model, num_heads)
x   = torch.randn(batch, seq_len, d_model)

out, weights = mha(x, x, x)   # self-attention: Q=K=V
print(f"MHA output shape:  {out.shape}")      # (4, 20, 512)
print(f"MHA weights shape: {weights.shape}")  # (4, 8, 20, 20)

MHA output shape:  torch.Size([4, 20, 512])
MHA weights shape: torch.Size([4, 8, 20, 20])


## Feed-Forward Block

From the paper: two linear layers with ReLU in between, expanding by 4x.
```
FFN(x) = max(0, x·W1 + b1)·W2 + b2
```

In [4]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)

## Encoder Block

One encoder layer = multi-head self-attention + feed-forward, each with a residual connection and layer norm.
```
x = LayerNorm(x + MultiHeadAttention(x, x, x))
x = LayerNorm(x + FFN(x))
```

In [5]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn    = MultiHeadAttention(d_model, num_heads)
        self.ff      = FeedForward(d_model, d_ff)
        self.norm1   = nn.LayerNorm(d_model)
        self.norm2   = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out, _ = self.attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))
        x = self.norm2(x + self.dropout(self.ff(x)))
        return x


# sanity check — does the shape hold through a full block?
block = EncoderBlock(d_model=512, num_heads=8, d_ff=2048)
x     = torch.randn(4, 20, 512)
out   = block(x)
print(f"Encoder block output: {out.shape}")  # (4, 20, 512)

Encoder block output: torch.Size([4, 20, 512])


## Compare to PyTorch's nn.MultiheadAttention

Quick check — does my implementation produce the same output shapes and reasonable attention weights?

In [6]:
torch_mha = nn.MultiheadAttention(embed_dim=512, num_heads=8, batch_first=True, bias=False)
x = torch.randn(4, 20, 512)

torch_out, torch_weights = torch_mha(x, x, x)
print(f"PyTorch MHA output:  {torch_out.shape}")

my_mha = MultiHeadAttention(512, 8)
my_out, my_weights = my_mha(x, x, x)
print(f"My MHA output:       {my_out.shape}")

# shapes match — values differ because weights are randomly initialised differently
print(f"\nShapes match: {torch_out.shape == my_out.shape}")

PyTorch MHA output:  torch.Size([4, 20, 512])
My MHA output:       torch.Size([4, 20, 512])

Shapes match: True


## Notes

Things I found interesting:
- The split/combine heads operation is just a reshape + transpose. It's not splitting the model into separate modules, it's one big matrix op that happens to be chunked.
- Pre-LN vs Post-LN matters more than I thought. The original paper uses Post-LN (norm after residual) but most modern implementations use Pre-LN because it's more stable. Worth reading into.
- The causal mask for autoregressive models is just setting future positions to -inf before softmax — clean.

Next: add positional encoding and stack N encoder blocks into a full encoder.